In [ ]:
from pathlib import Path
import shutil, os
# Reuse the pinned source and cached downloads on the attached Volume.
shutil.copytree("/mnt/m3diff/baseline_reproduction_v1/repo", "/root/M3Diff", ignore=shutil.ignore_patterns(".git", "__pycache__"))


In [ ]:
import os, sys, json, shutil, subprocess, glob, time
VOL   = "/mnt/m3diff"
REPO  = "/root/M3Diff"
VENV  = "/root/m3diff-venv"
PY    = f"{VENV}/bin/python"
CKPT  = "/root/ckpt/M3Diff"
SYN   = f"{VOL}/synth"            # Notebook 1
FT    = f"{VOL}/ft"               # Notebook này
CODE  = f"{FT}/code"
OUT   = f"{FT}/m3diff-ft-lora"    # tên thư mục phải chứa 'm3diff' và 'lora' để repo nạp đúng kiểu LoRA
EVAL  = f"{FT}/eval"
OBJ_ALL   = f"{FT}/objects_all"
TRAIN_SYN = f"{FT}/train_synth"
CAMTEST   = f"{FT}/camtest"
MIX       = f"{FT}/train_mix.json"
QUANT, DTYPE = "none", "bfloat16"

# Siêu tham số
SCALE    = 1.0        # nhân số cặp train tổng hợp (1.0 ≈ 2.700 cặp)
N_REPLAY = 2700       # số câu OmniDiff train gốc trộn vào
LORA_R, LORA_ALPHA = 64, 128
LR, PROJ_LR, EPOCHS = 4e-5, 1e-5, 1
HF_TOKEN = os.environ.get("HF_TOKEN", "")
os.environ["MPLBACKEND"] = "Agg"
os.environ["TQDM_MININTERVAL"] = "20"
for d in [CODE, EVAL, f"{FT}/results"]:
    os.makedirs(d, exist_ok=True)

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                     capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "KHÔNG CÓ GPU")
mem = int(gpu.split(",")[-1]) if gpu else 0
BATCH = 4 if mem >= 70000 else (2 if mem >= 39000 else 1)
ACCUM = 16 // BATCH
print(f"batch/GPU = {BATCH}, tích luỹ gradient = {ACCUM} (batch hiệu dụng 16)")
if mem < 39000:
    print("!! GPU dưới 40GB: train có thể hết bộ nhớ. Nên chọn A100.")

need = {"code Notebook 1": f"{SYN}/code/make_synth.py", "bộ test tổng hợp": f"{SYN}/data/manifest.jsonl",
        "câu model gốc trên bộ tổng hợp": f"{SYN}/data/answers.jsonl", "điểm model gốc (per_pair)": f"{SYN}/results/per_pair.csv",
        "vật thể test": f"{SYN}/objects/objects.csv", "kết quả gốc Real": f"{VOL}/results/OmniDiff_Real/OmniDiff_Real.jsonl"}
for k, v in need.items():
    print(("OK    " if os.path.exists(v) else "THIẾU ") + f"{k}: {v}")
print(("OK    " if os.path.exists(f"{VOL}/results/OmniDiff_Render/OmniDiff_Render.jsonl") else "(không có, bỏ qua Render) ")
      + "kết quả gốc Render")

In [ ]:
%pip install -q uv
UV = shutil.which("uv") or os.path.join(os.path.dirname(sys.executable), "uv")

if not os.path.exists(REPO):
    !git clone -q https://github.com/Yuan-Liu-01/M3Diff.git {REPO}
if not os.path.exists(PY):
    !{UV} venv {VENV} --python 3.10 -q

%cd {REPO}
!{UV} pip install -q --python {PY} -e ".[eval]" "numpy<2" accelerate==0.29.1 bitsandbytes==0.43.3 install-jdk
!{PY} -c "import torch, transformers, numpy; print('torch', torch.__version__, '| transformers', transformers.__version__, '| numpy', numpy.__version__); print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0), '|', round(torch.cuda.get_device_properties(0).total_memory/2**30, 1), 'GB')"
!{UV} pip install -q --python {PY} peft==0.4.0 "numpy<2" "opencv-python-headless<4.12" pandas scipy matplotlib pycocotools
!{PY} -c "import peft, numpy, cv2; print('peft', peft.__version__, '| numpy', numpy.__version__, '| cv2', cv2.__version__)"

In [ ]:
!{UV} pip install -q --python {PY} setuptools
!{PY} -c "import peft, numpy, cv2; print('peft', peft.__version__, '| numpy', numpy.__version__, '| cv2', cv2.__version__)"

In [ ]:
import hashlib
os.makedirs('/root/ckpt', exist_ok=True)
os.symlink('/mnt/m3diff/baseline_cache/M3Diff', CKPT)
os.makedirs(f'{REPO}/datasets', exist_ok=True)
os.symlink('/mnt/m3diff/baseline_cache/OmniDiff', f'{REPO}/datasets/OmniDiff')
assert os.path.isfile(f'{CKPT}/generation_config.json')
assert os.path.isfile(f'{CAMTEST}/questions.json')
print('Bước 3: dùng checkpoint và ảnh đã cache trên Volume; không tải lại.')


In [ ]:
path = f"{REPO}/llava/eval/model_vqa_qwen_OmniDiff_Real.py"
src = open(path).read()

# --- Patch 1: nén model cho vừa T4 ---
old1 = """    tokenizer, model, image_processor, _ = load_pretrained_model(
        model_path,
        args.model_base,
        model_name,
        use_flash_attn=args.flash_attention,
        torch_dtype=torch_dtype,
    )"""
new1 = """    quant_mode = os.environ.get("M3DIFF_QUANT", "8bit")
    skip = ["vision_tower", "vision_resampler", "mm_projector", "lm_head"]
    extra = {}
    if quant_mode == "8bit":
        extra["quantization_config"] = transformers.BitsAndBytesConfig(
            load_in_8bit=True, llm_int8_skip_modules=skip)
    elif quant_mode == "4bit":
        extra["quantization_config"] = transformers.BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
            llm_int8_skip_modules=skip)
    print(f"[patch] quant_mode={quant_mode}", flush=True)
    tokenizer, model, image_processor, _ = load_pretrained_model(
        model_path,
        args.model_base,
        model_name,
        use_flash_attn=args.flash_attention,
        torch_dtype=torch_dtype,
        device_map={"": 0},
        **extra,
    )"""

# --- Patch 2: chạy tiếp từ chỗ dừng + ghi từng câu ---
old2 = '    ans_file = open(answers_file, "w")\n'
new2 = """    done_lines = []
    if os.path.exists(answers_file):
        for l in open(answers_file):
            try:
                json.loads(l)
                done_lines.append(l.strip())
            except Exception:
                pass
    done_ids = {json.loads(l)["id"] for l in done_lines}
    with open(answers_file, "w") as f:
        f.write("".join(l + "\\n" for l in done_lines))
    questions = [q for q in questions if q["id"] not in done_ids]
    print(f"[patch] resume: {len(done_ids)} done, {len(questions)} left", flush=True)
    ans_file = open(answers_file, "a")
"""
old3 = '        ans_file.flush()\n'
new3 = '        ans_file.close()\n        ans_file = open(answers_file, "a")\n'

if "M3DIFF_QUANT" not in src:
    assert old1 in src, "Không tìm thấy đoạn patch 1"
    src = src.replace(old1, new1)
if "[patch] resume" not in src:
    assert old2 in src and old3 in src, "Không tìm thấy đoạn patch 2"
    src = src.replace(old2, new2).replace(old3, new3)
open(path, "w").write(src)
print("Patch OK:", "M3DIFF_QUANT" in src, "[patch] resume" in src)

In [ ]:
from IPython.display import Image, display
import pandas as pd
os.makedirs(f"{REPO}/datasets", exist_ok=True)
link = f"{REPO}/datasets/ft_synth"          # ảnh train tổng hợp nằm trên Volume, nối vào thư mục datasets
if os.path.isdir(TRAIN_SYN) and not os.path.islink(link):
    os.symlink(TRAIN_SYN, link)

def run_eval(qfile, answers, image_folder, lora):
    mp = f"--model-path {OUT} --model-base {CKPT}" if lora else f"--model-path {CKPT}"
    cmd = (f"cd {REPO} && PYTHONWARNINGS=ignore M3DIFF_QUANT={QUANT} {PY} llava/eval/model_vqa_qwen_OmniDiff_Real.py "
           f"{mp} --question-file {qfile} --answers-file {answers} --image-folder {image_folder} "
           f"--num-chunks 1 --chunk-idx 0 --temperature 0 --torch-dtype {DTYPE} --no-flash-attention")
    n_total = len(json.load(open(qfile)))
    done = sum(1 for _ in open(answers)) if os.path.exists(answers) else 0
    if done >= n_total:
        print(f"Đã đủ {done}/{n_total}: {answers}"); return
    t0 = time.time()
    rc = subprocess.run(cmd, shell=True).returncode
    done = sum(1 for _ in open(answers)) if os.path.exists(answers) else 0
    print(f"-> {done}/{n_total} câu, {(time.time()-t0)/60:.1f} phút, mã thoát {rc}")

MDP_ARGS = '{"num_refinement_layers":2,"feature_dim":1152,"ffn_hidden_dim":4608,"attention_downsample_rate":1}'

def train_args(out_dir, extra=()):
    return [PY, f"{CODE}/ft_train.py",
        "--model_name_or_path", CKPT, "--version", "qwen_2", "--output_dir", out_dir,
        "--data_path", MIX, "--image_folder", f"{REPO}/datasets", "--image_aspect_ratio", "pad",
        "--vision_tower", "google/siglip-so400m-patch14-384", "--mm_vision_select_layer", "[-11,-8,-5,-2]",
        "--unfreeze_mm_vision_tower", "False", "--mm_projector_type", "mlp2x_gelu", "--mm_projector_lr", str(PROJ_LR),
        "--mm_patch_merge_type", "flat", "--mm_resampler_type", "mdp", "--mdp_args", MDP_ARGS,
        "--tune_mm_vision_resampler", "True", "--mm_use_im_start_end", "False", "--mm_use_im_patch_token", "False",
        "--lora_enable", "True", "--lora_r", str(LORA_R), "--lora_alpha", str(LORA_ALPHA),
        "--num_train_epochs", str(EPOCHS), "--per_device_train_batch_size", str(BATCH),
        "--gradient_accumulation_steps", str(ACCUM), "--learning_rate", str(LR), "--weight_decay", "0.",
        "--warmup_ratio", "0.03", "--lr_scheduler_type", "cosine", "--logging_steps", "5",
        "--model_max_length", "4096", "--gradient_checkpointing", "True", "--group_by_modality_length", "True",
        "--dataloader_num_workers", "4", "--lazy_preprocess", "True", "--bf16", "True", "--tf32", "True",
        "--report_to", "none", *extra]

def run(args):
    from collections import deque
    env = dict(os.environ, ATTN="sdpa", PYTHONWARNINGS="ignore", TOKENIZERS_PARALLELISM="false")
    t0, tail = time.time(), deque(maxlen=40)
    p = subprocess.Popen(args, cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    keep = ("'loss'", "train_runtime", "out of memory", "resume", "it/s", "s/it", "Adding LoRA")
    for line in p.stdout:
        tail.append(line.rstrip())
        if any(k in line for k in keep):
            print(line.rstrip()[-300:], flush=True)
    p.wait()
    if p.returncode != 0:
        print("\n----- LỖI: 40 dòng cuối -----\n" + "\n".join(tail))
    print(f"[xong] mã thoát {p.returncode}, {(time.time()-t0)/60:.1f} phút")
    return p.returncode
print("OK")

In [ ]:
# Mandatory stop instead of allowing Step 11 to start training.
assert os.path.isfile(f'{OUT}/adapter_model.bin') or os.path.isfile(f'{OUT}/adapter_model.safetensors'), 'STOP: chưa có checkpoint đã train'
state=json.load(open(f'{OUT}/trainer_state.json'))
assert state['global_step']==state['max_steps']==338, 'STOP: training not complete'
if not os.path.islink(f"{REPO}/datasets/ft_synth"):
    os.symlink(TRAIN_SYN, f"{REPO}/datasets/ft_synth")
if os.path.exists(f"{OUT}/adapter_model.bin") or os.path.exists(f"{OUT}/adapter_model.safetensors"):
    print("Đã train xong:", OUT)
else:
    rc = run(train_args(OUT, ["--save_strategy", "steps", "--save_steps", "50", "--save_total_limit", "1"]))

# Đảm bảo khi nạp lại, vision tower lấy đúng trọng số đã fine-tune của M3Diff gốc (nạp sớm từ checkpoint)
cfg_p = f"{OUT}/config.json"
if os.path.exists(cfg_p):
    cfg = json.load(open(cfg_p)); cfg["unfreeze_mm_vision_tower"] = True
    json.dump(cfg, open(cfg_p, "w"), indent=2)
    print("Đã lưu model:", sorted(os.listdir(OUT)))

In [ ]:
# Fail fast when an eval subprocess fails or output IDs are incomplete.
_run_eval=run_eval
def run_eval(qfile, answers, image_folder, lora):
    start=time.time()
    _run_eval(qfile, answers, image_folder, lora)
    q=json.load(open(qfile)); rows=[json.loads(l) for l in open(answers) if l.strip()]
    assert len(rows)==len(q) and {str(r['id']) for r in rows}=={str(r['id']) for r in q}, 'Incomplete evaluation: STOP'
    assert all(r['caption'].strip() for r in rows)
    timings.setdefault('evals',[]).append({'questions':qfile,'answers':answers,'count':len(rows),'wall_seconds':time.time()-start})
    json.dump(timings,open(f'{RUN}/timings.json','w'),indent=2)
RUN='/mnt/m3diff/r64_synth_eval_20261102'
os.makedirs(RUN,exist_ok=True)
timings={'gpu':gpu,'source':'m3diff-nb-finetune.ipynb','evals':[]}


In [ ]:
selection={'seed': 20261102, 'method': 'stratified random 6 per camera-only condition;8 per object condition; fixed before synth600 results', 'rows': [{'pair_id': 'cam_remove_0027', 'seed': 700100237, 'image_A': 'images/cam_remove_0027_A.jpg', 'image_B': 'images/cam_remove_0027_B.jpg', 'caption': 'The shooting perspective has shifted to the left. The orange bottle that was in the center of the scene has been removed.', 'prompt': 'Explain the disparities between the first and second image.  <image><image>Difference:', 'level': 0, 'level_value': None, 'bg_source': 'Real/test/images/real_test_000124.jpg', 'cond': 'cam_remove', 'category': 'bottle', 'obj_id': 89064, 'obj_change': 'remove', 'camera': 'pan', 'camera_value': -0.1427, 'camera_dir': 'left'}, {'pair_id': 'cam_remove_0016', 'seed': 700100226, 'image_A': 'images/cam_remove_0016_A.jpg', 'image_B': 'images/cam_remove_0016_B.jpg', 'caption': 'The camera has rotated clockwise. The black handbag in the center of the scene has disappeared.', 'prompt': 'Point out the variations between the two images.  <image><image>Difference:', 'level': 0, 'level_value': None, 'bg_source': 'Render/test/images/render_test_outdoor_000061.jpg', 'cond': 'cam_remove', 'category': 'handbag', 'obj_id': 1178566, 'obj_change': 'remove', 'camera': 'rotate', 'camera_value': 11.117, 'camera_dir': 'clockwise'}, {'pair_id': 'pan_L3_07', 'seed': 700100172, 'image_A': 'images/pan_L3_07_A.jpg', 'image_B': 'images/pan_L3_07_B.jpg', 'caption': 'The camera has panned to the left.', 'prompt': 'Explain the disparities between the first and second image.  <image><image>Difference:', 'level': 3, 'level_value': 0.22, 'bg_source': 'Real/test/images/real_test_000124.jpg', 'cond': 'pan', 'camera': 'pan', 'camera_value': -0.22, 'camera_dir': 'left'}, {'pair_id': 'rotate_L3_09', 'seed': 700100129, 'image_A': 'images/rotate_L3_09_A.jpg', 'image_B': 'images/rotate_L3_09_B.jpg', 'caption': 'The camera has rotated clockwise.', 'prompt': 'What has changed from the first image to the second?  <image><image>Difference:', 'level': 3, 'level_value': 15, 'bg_source': 'Real/test/images/real_test_000147.jpg', 'cond': 'rotate', 'camera': 'rotate', 'camera_value': 15.0, 'camera_dir': 'clockwise'}, {'pair_id': 'cam_add_0021', 'seed': 700100201, 'image_A': 'images/cam_add_0021_A.jpg', 'image_B': 'images/cam_add_0021_B.jpg', 'caption': 'The camera has zoomed out, so more of the scene is visible. A brown teddy bear has been added in the center of the scene.', 'prompt': 'What distinguishes these two images?  <image><image>Difference:', 'level': 0, 'level_value': None, 'bg_source': 'Real/test/images/real_test_000166.jpg', 'cond': 'cam_add', 'category': 'teddy bear', 'obj_id': 1995385, 'obj_change': 'add', 'camera': 'zoom', 'camera_value': 0.7822, 'camera_dir': 'out'}, {'pair_id': 'cam_remove_0021', 'seed': 700100231, 'image_A': 'images/cam_remove_0021_A.jpg', 'image_B': 'images/cam_remove_0021_B.jpg', 'caption': 'The camera has zoomed in, making objects appear larger. The white car that was in the center of the scene has been removed.', 'prompt': 'What modifications can be observed between these two pictures?  <image><image>Difference:', 'level': 0, 'level_value': None, 'bg_source': 'Real/test/images/real_test_000159.jpg', 'cond': 'cam_remove', 'category': 'car', 'obj_id': 354293, 'obj_change': 'remove', 'camera': 'zoom', 'camera_value': 1.2386, 'camera_dir': 'in'}, {'pair_id': 'zoom_in_L1_04', 'seed': 700100004, 'image_A': 'images/zoom_in_L1_04_A.jpg', 'image_B': 'images/zoom_in_L1_04_B.jpg', 'caption': 'The camera has slightly zoomed in, making objects appear larger.', 'prompt': 'Point out the variations between the two images.  <image><image>Difference:', 'level': 1, 'level_value': 1.12, 'bg_source': 'Real/test/images/real_test_000186.jpg', 'cond': 'zoom_in', 'camera': 'zoom', 'camera_value': 1.12, 'camera_dir': 'in'}, {'pair_id': 'cam_add_0025', 'seed': 700100205, 'image_A': 'images/cam_add_0025_A.jpg', 'image_B': 'images/cam_add_0025_B.jpg', 'caption': 'The shooting perspective has shifted to the right. A red teddy bear has been added in the center of the scene.', 'prompt': "What's the difference between 2 images?  <image><image>Difference:", 'level': 0, 'level_value': None, 'bg_source': 'Render/test/images/render_test_outdoor_000001.jpg', 'cond': 'cam_add', 'category': 'teddy bear', 'obj_id': 1162599, 'obj_change': 'add', 'camera': 'pan', 'camera_value': 0.2206, 'camera_dir': 'right'}, {'pair_id': 'pan_L1_12', 'seed': 700100147, 'image_A': 'images/pan_L1_12_A.jpg', 'image_B': 'images/pan_L1_12_B.jpg', 'caption': 'The camera has slightly panned to the left.', 'prompt': 'What has changed from the first image to the second?  <image><image>Difference:', 'level': 1, 'level_value': 0.06, 'bg_source': 'Render/test/images/render_test_indoor_000052.jpg', 'cond': 'pan', 'camera': 'pan', 'camera_value': -0.06, 'camera_dir': 'left'}, {'pair_id': 'pan_L3_06', 'seed': 700100171, 'image_A': 'images/pan_L3_06_A.jpg', 'image_B': 'images/pan_L3_06_B.jpg', 'caption': 'The camera has panned to the left.', 'prompt': 'Can you highlight the differences in these two pictures?  <image><image>Difference:', 'level': 3, 'level_value': 0.22, 'bg_source': 'Render/test/images/render_test_indoor_000007.jpg', 'cond': 'pan', 'camera': 'pan', 'camera_value': -0.22, 'camera_dir': 'left'}, {'pair_id': 'cam_add_0016', 'seed': 700100196, 'image_A': 'images/cam_add_0016_A.jpg', 'image_B': 'images/cam_add_0016_B.jpg', 'caption': 'The shooting perspective has shifted to the right. A new blue cup is now present in the center of the scene.', 'prompt': "What's the difference between 2 images?  <image><image>Difference:", 'level': 0, 'level_value': None, 'bg_source': 'Real/test/images/real_test_000149.jpg', 'cond': 'cam_add', 'category': 'cup', 'obj_id': 677591, 'obj_change': 'add', 'camera': 'pan', 'camera_value': 0.1891, 'camera_dir': 'right'}, {'pair_id': 'zoom_in_L2_09', 'seed': 700100024, 'image_A': 'images/zoom_in_L2_09_A.jpg', 'image_B': 'images/zoom_in_L2_09_B.jpg', 'caption': 'The camera has zoomed in, making objects appear larger.', 'prompt': 'What modifications can be observed between these two pictures?  <image><image>Difference:', 'level': 2, 'level_value': 1.25, 'bg_source': 'Render/test/images/render_test_indoor_000052.jpg', 'cond': 'zoom_in', 'camera': 'zoom', 'camera_value': 1.25, 'camera_dir': 'in'}, {'pair_id': 'cam_remove_0000', 'seed': 700100210, 'image_A': 'images/cam_remove_0000_A.jpg', 'image_B': 'images/cam_remove_0000_B.jpg', 'caption': 'The shooting perspective has shifted to the right. The yellow apple that was in the center of the scene has been removed.', 'prompt': 'Can you highlight the differences in these two pictures?  <image><image>Difference:', 'level': 0, 'level_value': None, 'bg_source': 'Real/test/images/real_test_000140.jpg', 'cond': 'cam_remove', 'category': 'apple', 'obj_id': 1048544, 'obj_change': 'remove', 'camera': 'pan', 'camera_value': 0.1886, 'camera_dir': 'right'}, {'pair_id': 'zoom_in_L3_06', 'seed': 700100036, 'image_A': 'images/zoom_in_L3_06_A.jpg', 'image_B': 'images/zoom_in_L3_06_B.jpg', 'caption': 'The shooting perspective has zoomed in.', 'prompt': 'Describe the contrast between these two images.  <image><image>Difference:', 'level': 3, 'level_value': 1.5, 'bg_source': 'Render/test/images/render_test_outdoor_000016.jpg', 'cond': 'zoom_in', 'camera': 'zoom', 'camera_value': 1.5, 'camera_dir': 'in'}, {'pair_id': 'zoom_out_L3_03', 'seed': 700100078, 'image_A': 'images/zoom_out_L3_03_A.jpg', 'image_B': 'images/zoom_out_L3_03_B.jpg', 'caption': 'The shooting perspective has zoomed out.', 'prompt': 'Describe the contrast between these two images.  <image><image>Difference:', 'level': 3, 'level_value': 1.5, 'bg_source': 'Render/test/images/render_test_outdoor_000017.jpg', 'cond': 'zoom_out', 'camera': 'zoom', 'camera_value': 0.6667, 'camera_dir': 'out'}, {'pair_id': 'rotate_L3_08', 'seed': 700100128, 'image_A': 'images/rotate_L3_08_A.jpg', 'image_B': 'images/rotate_L3_08_B.jpg', 'caption': 'The camera has rotated clockwise.', 'prompt': 'What modifications can be observed between these two pictures?  <image><image>Difference:', 'level': 3, 'level_value': 15, 'bg_source': 'Real/test/images/real_test_000072.jpg', 'cond': 'rotate', 'camera': 'rotate', 'camera_value': 15.0, 'camera_dir': 'clockwise'}, {'pair_id': 'cam_add_0017', 'seed': 700100197, 'image_A': 'images/cam_add_0017_A.jpg', 'image_B': 'images/cam_add_0017_B.jpg', 'caption': 'The shooting perspective has zoomed in. A new brown book is now present in the center of the scene.', 'prompt': 'Can you highlight the differences in these two pictures?  <image><image>Difference:', 'level': 0, 'level_value': None, 'bg_source': 'Render/test/images/render_test_indoor_000060.jpg', 'cond': 'cam_add', 'category': 'book', 'obj_id': 1662457, 'obj_change': 'add', 'camera': 'zoom', 'camera_value': 1.2929, 'camera_dir': 'in'}, {'pair_id': 'zoom_out_L2_10', 'seed': 700100070, 'image_A': 'images/zoom_out_L2_10_A.jpg', 'image_B': 'images/zoom_out_L2_10_B.jpg', 'caption': 'The shooting perspective has zoomed out.', 'prompt': "What's the difference between 2 images?  <image><image>Difference:", 'level': 2, 'level_value': 1.25, 'bg_source': 'Render/test/images/render_test_outdoor_000058.jpg', 'cond': 'zoom_out', 'camera': 'zoom', 'camera_value': 0.8, 'camera_dir': 'out'}, {'pair_id': 'zoom_out_L3_11', 'seed': 700100086, 'image_A': 'images/zoom_out_L3_11_A.jpg', 'image_B': 'images/zoom_out_L3_11_B.jpg', 'caption': 'The camera has zoomed out, so more of the scene is visible.', 'prompt': "What's the difference between 2 images?  <image><image>Difference:", 'level': 3, 'level_value': 1.5, 'bg_source': 'Real/test/images/real_test_000075.jpg', 'cond': 'zoom_out', 'camera': 'zoom', 'camera_value': 0.6667, 'camera_dir': 'out'}, {'pair_id': 'zoom_in_L1_07', 'seed': 700100007, 'image_A': 'images/zoom_in_L1_07_A.jpg', 'image_B': 'images/zoom_in_L1_07_B.jpg', 'caption': 'The camera has slightly zoomed in, making objects appear larger.', 'prompt': 'What has changed from the first image to the second?  <image><image>Difference:', 'level': 1, 'level_value': 1.12, 'bg_source': 'Render/test/images/render_test_indoor_000024.jpg', 'cond': 'zoom_in', 'camera': 'zoom', 'camera_value': 1.12, 'camera_dir': 'in'}, {'pair_id': 'cam_add_0002', 'seed': 700100182, 'image_A': 'images/cam_add_0002_A.jpg', 'image_B': 'images/cam_add_0002_B.jpg', 'caption': 'The camera has rotated counterclockwise. A gray clock has been added in the center of the scene.', 'prompt': 'What has changed from the first image to the second?  <image><image>Difference:', 'level': 0, 'level_value': None, 'bg_source': 'Render/test/images/render_test_indoor_000032.jpg', 'cond': 'cam_add', 'category': 'clock', 'obj_id': 335955, 'obj_change': 'add', 'camera': 'rotate', 'camera_value': -7.5615, 'camera_dir': 'counterclockwise'}, {'pair_id': 'cam_remove_0014', 'seed': 700100224, 'image_A': 'images/cam_remove_0014_A.jpg', 'image_B': 'images/cam_remove_0014_B.jpg', 'caption': 'The camera has slightly panned to the right. The white book in the center of the scene has disappeared.', 'prompt': 'Explain the disparities between the first and second image.  <image><image>Difference:', 'level': 0, 'level_value': None, 'bg_source': 'Real/test/images/real_test_000131.jpg', 'cond': 'cam_remove', 'category': 'book', 'obj_id': 1137084, 'obj_change': 'remove', 'camera': 'pan', 'camera_value': 0.0629, 'camera_dir': 'right'}, {'pair_id': 'pan_L2_13', 'seed': 700100163, 'image_A': 'images/pan_L2_13_A.jpg', 'image_B': 'images/pan_L2_13_B.jpg', 'caption': 'The camera has panned to the left.', 'prompt': 'Point out the variations between the two images.  <image><image>Difference:', 'level': 2, 'level_value': 0.12, 'bg_source': 'Render/test/images/render_test_outdoor_000050.jpg', 'cond': 'pan', 'camera': 'pan', 'camera_value': -0.12, 'camera_dir': 'left'}, {'pair_id': 'cam_add_0003', 'seed': 700100183, 'image_A': 'images/cam_add_0003_A.jpg', 'image_B': 'images/cam_add_0003_B.jpg', 'caption': 'The camera has panned to the left. An new orange teddy bear is now present in the center of the scene.', 'prompt': 'Can you highlight the differences in these two pictures?  <image><image>Difference:', 'level': 0, 'level_value': None, 'bg_source': 'Render/test/images/render_test_indoor_000038.jpg', 'cond': 'cam_add', 'category': 'teddy bear', 'obj_id': 1159557, 'obj_change': 'add', 'camera': 'pan', 'camera_value': -0.201, 'camera_dir': 'left'}, {'pair_id': 'zoom_in_L3_10', 'seed': 700100040, 'image_A': 'images/zoom_in_L3_10_A.jpg', 'image_B': 'images/zoom_in_L3_10_B.jpg', 'caption': 'The shooting perspective has zoomed in.', 'prompt': 'Identify the alterations between these two images.  <image><image>Difference:', 'level': 3, 'level_value': 1.5, 'bg_source': 'Real/test/images/real_test_000060.jpg', 'cond': 'zoom_in', 'camera': 'zoom', 'camera_value': 1.5, 'camera_dir': 'in'}, {'pair_id': 'rotate_L2_14', 'seed': 700100119, 'image_A': 'images/rotate_L2_14_A.jpg', 'image_B': 'images/rotate_L2_14_B.jpg', 'caption': 'The camera has rotated counterclockwise.', 'prompt': 'What modifications can be observed between these two pictures?  <image><image>Difference:', 'level': 2, 'level_value': 7, 'bg_source': 'Render/test/images/render_test_outdoor_000059.jpg', 'cond': 'rotate', 'camera': 'rotate', 'camera_value': -7.0, 'camera_dir': 'counterclockwise'}, {'pair_id': 'rotate_L3_05', 'seed': 700100125, 'image_A': 'images/rotate_L3_05_A.jpg', 'image_B': 'images/rotate_L3_05_B.jpg', 'caption': 'The shooting angle has tilted clockwise.', 'prompt': "What's the difference between 2 images?  <image><image>Difference:", 'level': 3, 'level_value': 15, 'bg_source': 'Real/test/images/real_test_000124.jpg', 'cond': 'rotate', 'camera': 'rotate', 'camera_value': 15.0, 'camera_dir': 'clockwise'}, {'pair_id': 'rotate_L1_11', 'seed': 700100101, 'image_A': 'images/rotate_L1_11_A.jpg', 'image_B': 'images/rotate_L1_11_B.jpg', 'caption': 'The shooting angle has tilted slightly counterclockwise.', 'prompt': 'Describe the contrast between these two images.  <image><image>Difference:', 'level': 1, 'level_value': 3, 'bg_source': 'Render/test/images/render_test_outdoor_000016.jpg', 'cond': 'rotate', 'camera': 'rotate', 'camera_value': -3.0, 'camera_dir': 'counterclockwise'}, {'pair_id': 'pan_L2_11', 'seed': 700100161, 'image_A': 'images/pan_L2_11_A.jpg', 'image_B': 'images/pan_L2_11_B.jpg', 'caption': 'The camera has panned to the left.', 'prompt': 'Describe the contrast between these two images.  <image><image>Difference:', 'level': 2, 'level_value': 0.12, 'bg_source': 'Real/test/images/real_test_000031.jpg', 'cond': 'pan', 'camera': 'pan', 'camera_value': -0.12, 'camera_dir': 'left'}, {'pair_id': 'cam_add_0020', 'seed': 700100200, 'image_A': 'images/cam_add_0020_A.jpg', 'image_B': 'images/cam_add_0020_B.jpg', 'caption': 'The camera has rotated clockwise. A brown teddy bear has appeared in the center of the scene.', 'prompt': 'What has changed from the first image to the second?  <image><image>Difference:', 'level': 0, 'level_value': None, 'bg_source': 'Real/test/images/real_test_000173.jpg', 'cond': 'cam_add', 'category': 'teddy bear', 'obj_id': 1162597, 'obj_change': 'add', 'camera': 'rotate', 'camera_value': 6.7267, 'camera_dir': 'clockwise'}, {'pair_id': 'zoom_out_L1_12', 'seed': 700100057, 'image_A': 'images/zoom_out_L1_12_A.jpg', 'image_B': 'images/zoom_out_L1_12_B.jpg', 'caption': 'The camera has slightly zoomed out, so more of the scene is visible.', 'prompt': 'Identify the alterations between these two images.  <image><image>Difference:', 'level': 1, 'level_value': 1.12, 'bg_source': 'Render/test/images/render_test_indoor_000008.jpg', 'cond': 'zoom_out', 'camera': 'zoom', 'camera_value': 0.8929, 'camera_dir': 'out'}, {'pair_id': 'zoom_out_L2_11', 'seed': 700100071, 'image_A': 'images/zoom_out_L2_11_A.jpg', 'image_B': 'images/zoom_out_L2_11_B.jpg', 'caption': 'The camera has zoomed out, so more of the scene is visible.', 'prompt': 'Identify the alterations between these two images.  <image><image>Difference:', 'level': 2, 'level_value': 1.25, 'bg_source': 'Render/test/images/render_test_indoor_000033.jpg', 'cond': 'zoom_out', 'camera': 'zoom', 'camera_value': 0.8, 'camera_dir': 'out'}, {'pair_id': 'cam_remove_0002', 'seed': 700100212, 'image_A': 'images/cam_remove_0002_A.jpg', 'image_B': 'images/cam_remove_0002_B.jpg', 'caption': 'The camera has rotated counterclockwise. The red apple in the center of the scene has disappeared.', 'prompt': 'Explain the disparities between the first and second image.  <image><image>Difference:', 'level': 0, 'level_value': None, 'bg_source': 'Real/test/images/real_test_000156.jpg', 'cond': 'cam_remove', 'category': 'apple', 'obj_id': 1049024, 'obj_change': 'remove', 'camera': 'rotate', 'camera_value': -9.6934, 'camera_dir': 'counterclockwise'}, {'pair_id': 'rotate_L1_12', 'seed': 700100102, 'image_A': 'images/rotate_L1_12_A.jpg', 'image_B': 'images/rotate_L1_12_B.jpg', 'caption': 'The shooting angle has tilted slightly clockwise.', 'prompt': 'What distinguishes these two images?  <image><image>Difference:', 'level': 1, 'level_value': 3, 'bg_source': 'Render/test/images/render_test_indoor_000008.jpg', 'cond': 'rotate', 'camera': 'rotate', 'camera_value': 3.0, 'camera_dir': 'clockwise'}, {'pair_id': 'zoom_out_L1_02', 'seed': 700100047, 'image_A': 'images/zoom_out_L1_02_A.jpg', 'image_B': 'images/zoom_out_L1_02_B.jpg', 'caption': 'The shooting perspective has slightly zoomed out.', 'prompt': 'Explain the disparities between the first and second image.  <image><image>Difference:', 'level': 1, 'level_value': 1.12, 'bg_source': 'Real/test/images/real_test_000077.jpg', 'cond': 'zoom_out', 'camera': 'zoom', 'camera_value': 0.8929, 'camera_dir': 'out'}, {'pair_id': 'cam_add_0001', 'seed': 700100181, 'image_A': 'images/cam_add_0001_A.jpg', 'image_B': 'images/cam_add_0001_B.jpg', 'caption': 'The shooting perspective has shifted to the right. A gray book has appeared in the center of the scene.', 'prompt': 'What has changed from the first image to the second?  <image><image>Difference:', 'level': 0, 'level_value': None, 'bg_source': 'Real/test/images/real_test_000086.jpg', 'cond': 'cam_add', 'category': 'book', 'obj_id': 1139235, 'obj_change': 'add', 'camera': 'pan', 'camera_value': 0.1019, 'camera_dir': 'right'}, {'pair_id': 'zoom_in_L1_10', 'seed': 700100010, 'image_A': 'images/zoom_in_L1_10_A.jpg', 'image_B': 'images/zoom_in_L1_10_B.jpg', 'caption': 'The camera has slightly zoomed in, making objects appear larger.', 'prompt': 'What distinguishes these two images?  <image><image>Difference:', 'level': 1, 'level_value': 1.12, 'bg_source': 'Render/test/images/render_test_indoor_000057.jpg', 'cond': 'zoom_in', 'camera': 'zoom', 'camera_value': 1.12, 'camera_dir': 'in'}, {'pair_id': 'cam_remove_0024', 'seed': 700100234, 'image_A': 'images/cam_remove_0024_A.jpg', 'image_B': 'images/cam_remove_0024_B.jpg', 'caption': 'The shooting angle has tilted counterclockwise. The black car in the center of the scene has disappeared.', 'prompt': 'Point out the variations between the two images.  <image><image>Difference:', 'level': 0, 'level_value': None, 'bg_source': 'Real/test/images/real_test_000132.jpg', 'cond': 'cam_remove', 'category': 'car', 'obj_id': 135318, 'obj_change': 'remove', 'camera': 'rotate', 'camera_value': -6.6004, 'camera_dir': 'counterclockwise'}, {'pair_id': 'cam_remove_0017', 'seed': 700100227, 'image_A': 'images/cam_remove_0017_A.jpg', 'image_B': 'images/cam_remove_0017_B.jpg', 'caption': 'The camera has rotated slightly counterclockwise. The blue clock in the center of the scene has disappeared.', 'prompt': 'Describe the contrast between these two images.  <image><image>Difference:', 'level': 0, 'level_value': None, 'bg_source': 'Render/test/images/render_test_indoor_000007.jpg', 'cond': 'cam_remove', 'category': 'clock', 'obj_id': 334181, 'obj_change': 'remove', 'camera': 'rotate', 'camera_value': -2.949, 'camera_dir': 'counterclockwise'}, {'pair_id': 'pan_L2_05', 'seed': 700100155, 'image_A': 'images/pan_L2_05_A.jpg', 'image_B': 'images/pan_L2_05_B.jpg', 'caption': 'The camera has panned to the right.', 'prompt': 'Explain the disparities between the first and second image.  <image><image>Difference:', 'level': 2, 'level_value': 0.12, 'bg_source': 'Real/test/images/real_test_000132.jpg', 'cond': 'pan', 'camera': 'pan', 'camera_value': 0.12, 'camera_dir': 'right'}], 'n': 40, 'rater': 'assistant, not independent human'}
json.dump(selection,open(f'{RUN}/REVIEW_SELECTION.json','w'),ensure_ascii=False,indent=2)
subprocess.run([PY,'-c',"from PIL import Image,ImageDraw,ImageFont\nfrom pathlib import Path\nimport json,textwrap\nroot=Path('/mnt/m3diff');out=root/'r64_synth_eval_20261102';sel=json.loads((out/'REVIEW_SELECTION.json').read_text())\nbase={r['id']:r['caption'] for r in [json.loads(l) for l in (root/'ft/eval/camtest_base.jsonl').read_text().splitlines() if l.strip()]}\nft={r['id']:r['caption'] for r in [json.loads(l) for l in (root/'ft/eval/camtest_ft.jsonl').read_text().splitlines() if l.strip()]}\ntry:font=ImageFont.truetype('DejaVuSans.ttf',19)\nexcept OSError:font=ImageFont.load_default()\nfor i in range(0,40,4):\n    canvas=Image.new('RGB',(1600,3000),'white');d=ImageDraw.Draw(canvas)\n    for j,row in enumerate(sel['rows'][i:i+4]):\n        y=j*750;pid=row['pair_id'];d.text((10,y+5),f'{i+j+1}. {pid} expected: '+row['caption'],fill='black',font=font)\n        for side,name in enumerate(['image_A','image_B']):\n            with Image.open(root/'ft/camtest'/row[name]) as im:\n                im=im.convert('RGB');im.thumbnail((780,390));canvas.paste(im,(side*800+10,y+40))\n        for side,cap in enumerate([base[pid],ft[pid]]):\n            lines=textwrap.wrap(('BASE: ' if side==0 else 'R64: ')+cap,width=74)\n            for k,line in enumerate(lines[:13]):d.text((side*800+10,y+440+k*22),line,fill='black',font=font)\n    canvas.save(out/f'review_board_{i//4:02d}.jpg',quality=90)\nprint('Saved10reviewboards')\n"],check=True)


In [ ]:
# Exactly synth600 only. No OmniDiff or camera rerun.
assert len(json.load(open(f'{SYN}/data/questions.json')))==600
run_eval(f'{SYN}/data/questions.json',f'{EVAL}/synth_ft.jsonl',f'{SYN}/data',lora=True)
for arm,answers in [('base',f'{SYN}/data/answers.jsonl'),('r64',f'{EVAL}/synth_ft.jsonl')]:
    subprocess.run([PY,f'{SYN}/code/score_captions.py','--root',f'{SYN}/data','--answers',answers,'--out',f'{RUN}/{arm}_scores','--n-manual','0'],check=True)
    shutil.copy2(answers,f'{RUN}/{arm}.jsonl')
shutil.copy2(f'{SYN}/data/manifest.jsonl',f'{RUN}/manifest.jsonl')
shutil.copy2(f'{SYN}/data/questions.json',f'{RUN}/questions.json')
json.dump({'complete':True,'synth_count':600,'full_omnidiff_run':False},open(f'{RUN}/SUCCESS.json','w'),indent=2)
print('DONE synth600 only; release GPU now.')
